# SO101 トマト山積みコンテナ掴み 学習（Colab / GPU用）2026-10-27農園データ

**事前準備（Colabを開く前に）**
1. Macで作った `so101_tomato_crate_1cam_100ep.zip`（`so101-blog/`直下）を、Googleドライブの `MyDrive/so101/` フォルダにアップロードしてください
2. Colab上部メニュー「ランタイム」→「ランタイムのタイプを変更」→ハードウェアアクセラレータを **GPU** にする（有料のコンピューティングユニットがあれば A100 か V100、無ければ T4）
3. 上から順にセルを実行（Shift+Enter）してください

**メモ**: 100エピソード全量版。40本版(act_tomato_crate_40ep)とのスケーリング比較用。学習条件は40本版と完全同一(100kステップ/batch8)。
タスク: 緑コンテナ(山積み)からトマトを掴んで持ち上げる(俯瞰1カメラ)。


In [ ]:
# GPUがちゃんと割り当てられているか確認
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# アップロードしたzipを解凍（パスが違う場合はここを修正してください）
DRIVE_ZIP = "/content/drive/MyDrive/so101/so101_tomato_crate_1cam_100ep.zip"

!mkdir -p /content/dataset
!unzip -q "$DRIVE_ZIP" -d /content/dataset
!ls /content/dataset

In [ ]:
# lerobotをMacと同じバージョンでインストール（[dataset]で映像処理用のavも一緒に入れる）
!pip install -q "lerobot[dataset,training]==0.6.0" accelerate
# Colab環境(Python3.13)のtorchaoが古くdiffusersが落ちる対策(2026-10-27実証済み)
!pip install -q -U torchao


In [ ]:
# 解凍したデータセットの実際のフォルダ名を自動検出
import glob

candidates = glob.glob("/content/dataset/so101_tomato_crate_1cam_*")
assert len(candidates) == 1, f"データセットフォルダが1つに絞れません: {candidates}"
DATASET_ROOT = candidates[0]
print("DATASET_ROOT =", DATASET_ROOT)

In [ ]:
# 本番学習（100,000ステップ）。100エピソード全部使用(除外なし)。
# output_dirを最初からGoogleドライブにして、5000ステップごとのチェックポイントが都度残るようにしている。
!lerobot-train \
  --dataset.repo_id=local/so101_tomato_crate_1cam \
  --dataset.root="$DATASET_ROOT" \
  --policy.type=act \
  --policy.device=cuda \
  --policy.push_to_hub=false \
  --output_dir=/content/drive/MyDrive/so101/train_output/act_tomato_crate_100ep \
  --job_name=act_tomato_crate_100ep \
  --steps=100000 \
  --batch_size=8 \
  --save_freq=5000 \
  --log_freq=50 \
  --wandb.enable=false


In [ ]:
# output_dirが最初からGoogleドライブなので、ここでのコピー作業は不要。
# 完了確認だけ行う（チェックポイント一覧が表示されればOK）
!ls -la /content/drive/MyDrive/so101/train_output/act_tomato_crate_100ep/checkpoints